In [0]:
from pyspark.sql.functions import col, when

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

loans_raw = spark.table("bronze.loans_raw")
borrowers_raw = spark.table("bronze.borrowers_raw")

loans_clean = (loans_raw
    .withColumn("loan_amnt", col("loan_amnt").cast("double"))
    .withColumn("int_rate", col("int_rate").cast("double"))
    .withColumn("is_default", when(col("loan_status").isin("Charged Off", "Default"), 1).otherwise(0))
    .na.drop(subset=["id", "loan_amnt", "grade"])
    .dropDuplicates(["id"]))

loans_clean.write.format("delta").mode("overwrite").saveAsTable("silver.loans_clean")

loans_joined = loans_clean.join(
    borrowers_raw \
        .withColumnRenamed("emp_length", "borrower_emp_length") \
        .withColumnRenamed("emp_title", "borrower_emp_title") \
        .withColumnRenamed("home_ownership", "borrower_home_ownership"),
    on="id", how="left")
loans_joined.write.format("delta").mode("overwrite").saveAsTable("silver.loans_joined")